In [1]:
import pandas as pd
import sqlite3
import os

# Load cleaned CSV
cleaned_path = r"C:\Users\SHRUTI VAISHNAV\OneDrive\Desktop\data science\fda-faers-adverse-event-analysis\data\cleaned\faers_cleaned_2023q1.csv"

merged = pd.read_csv(cleaned_path)
print("Data loaded successfully.")
print("Shape:", merged.shape)
print("Columns:", merged.columns.tolist())

Data loaded successfully.
Shape: (1052002, 13)
Columns: ['primaryid', 'age', 'sex', 'reporter_country', 'occp_cod', 'age_group', 'drugname', 'role_cod', 'route', 'reaction', 'outc_cod', 'outcome_label', 'is_fatal']


In [2]:
# Create SQLite database in SQL folder
db_path = r"C:\Users\SHRUTI VAISHNAV\OneDrive\Desktop\data science\fda-faers-adverse-event-analysis\SQL\faers.db"

conn = sqlite3.connect(db_path)

# Load dataframe into SQLite as table named 'faers'
merged.to_sql('faers', conn, if_exists='replace', index=False)

print("Database created successfully.")
print("Table 'faers' loaded with", len(merged), "rows.")

conn.close()

Database created successfully.
Table 'faers' loaded with 1052002 rows.


In [3]:
conn = sqlite3.connect(db_path)

q1 = """
SELECT drugname,
       COUNT(*) AS total_reports,
       SUM(is_fatal) AS fatal_cases,
       ROUND(100.0 * SUM(is_fatal) / COUNT(*), 2) AS fatality_rate_pct
FROM faers
GROUP BY drugname
ORDER BY total_reports DESC
LIMIT 20;
"""

df_q1 = pd.read_sql_query(q1, conn)
print("Query 1 — Top 20 Drugs by Report Count:")
print(df_q1.to_string(index=False))

conn.close()

Query 1 — Top 20 Drugs by Report Count:
              drugname  total_reports  fatal_cases  fatality_rate_pct
            INFLIXIMAB          28678          262               0.91
           VEDOLIZUMAB          24054          164               0.68
          METHOTREXATE          22536          884               3.92
              COSENTYX          19937          786               3.94
                HUMIRA          15779          389               2.47
HUMAN IMMUNOGLOBULIN G          14683          397               2.70
              REMICADE          12645          173               1.37
               ACTEMRA          12079          497               4.11
               OCREVUS          11289           59               0.52
         CIPROFLOXACIN          10554          203               1.92
 SANDOSTATIN LAR DEPOT          10100          246               2.44
            XELJANZ XR           9859          173               1.75
                XOLAIR           8702           63

In [4]:
conn = sqlite3.connect(db_path)

q2 = """
SELECT drugname,
       COUNT(*) AS total_reports,
       SUM(is_fatal) AS fatal_cases,
       ROUND(100.0 * SUM(is_fatal) / COUNT(*), 2) AS fatality_rate_pct
FROM faers
GROUP BY drugname
HAVING total_reports >= 1000
ORDER BY fatality_rate_pct DESC
LIMIT 20;
"""

df_q2 = pd.read_sql_query(q2, conn)
print("Query 2 — Top 20 Drugs by Fatality Rate (min 1000 reports):")
print(df_q2.to_string(index=False))

conn.close()

Query 2 — Top 20 Drugs by Fatality Rate (min 1000 reports):
        drugname  total_reports  fatal_cases  fatality_rate_pct
FENTANYL CITRATE           1143          510              44.62
        FENTANYL           3607         1479              41.00
         ENHERTU           1072          314              29.29
   ACETAMINOPHEN           7748         2134              27.54
      VENETOCLAX           4584         1087              23.71
       BUPROPION           1528          340              22.25
       VENCLEXTA           3552          779              21.93
        TAFINLAR           1474          320              21.71
        NUPLAZID           4090          880              21.52
       TECENTRIQ           1297          273              21.05
          PADCEV           1100          209              19.00
         ELIQUIS           5320         1001              18.82
    VORICONAZOLE           1212          227              18.73
    CAPECITABINE           2101          372

In [5]:
conn = sqlite3.connect(db_path)

q3 = """
SELECT age_group,
       COUNT(*) AS total_reports,
       SUM(is_fatal) AS fatal_cases,
       ROUND(100.0 * SUM(is_fatal) / COUNT(*), 2) AS fatality_rate_pct
FROM faers
GROUP BY age_group
ORDER BY fatality_rate_pct DESC;
"""

df_q3 = pd.read_sql_query(q3, conn)
print("Query 3 — Fatality Rate by Age Group:")
print(df_q3.to_string(index=False))

conn.close()

Query 3 — Fatality Rate by Age Group:
age_group  total_reports  fatal_cases  fatality_rate_pct
      65+         361966        33716               9.31
    50-64         300331        18883               6.29
     0-17          63141         3963               6.28
    18-34         113285         6389               5.64
    35-49         213279        11888               5.57


In [6]:
conn = sqlite3.connect(db_path)

q4 = """
SELECT reaction,
       COUNT(*) AS report_count,
       SUM(is_fatal) AS fatal_count,
       ROUND(100.0 * SUM(is_fatal) / COUNT(*), 2) AS fatality_rate_pct
FROM faers
GROUP BY reaction
ORDER BY report_count DESC
LIMIT 20;
"""

df_q4 = pd.read_sql_query(q4, conn)
print("Query 4 — Top 20 Reactions with Fatality Rate:")
print(df_q4.to_string(index=False))

conn.close()

Query 4 — Top 20 Reactions with Fatality Rate:
                  reaction  report_count  fatal_count  fatality_rate_pct
             Off Label Use         20649         1522               7.37
                     Death         14560         8978              61.66
          Drug Ineffective         13189         1226               9.30
                   Fatigue         11707          406               3.47
                 Diarrhoea         10937          516               4.72
                    Nausea          9765          423               4.33
                  Dyspnoea          9117          505               5.54
                      Pain          8999          244               2.71
                 Pneumonia          8411          774               9.20
                  Covid-19          8370          464               5.54
                   Pyrexia          7954          465               5.85
                  Headache          7893          199               2.52
    

In [7]:
conn = sqlite3.connect(db_path)

q5 = """
SELECT drugname,
       reaction,
       COUNT(*) AS pair_count,
       SUM(is_fatal) AS fatal_count,
       ROUND(100.0 * SUM(is_fatal) / COUNT(*), 2) AS fatality_rate_pct
FROM faers
GROUP BY drugname, reaction
HAVING pair_count >= 500
ORDER BY pair_count DESC
LIMIT 25;
"""

df_q5 = pd.read_sql_query(q5, conn)
print("Query 5 — Top Drug-Reaction Pairs (min 500 reports):")
print(df_q5.to_string(index=False))

conn.close()

Query 5 — Top Drug-Reaction Pairs (min 500 reports):
     drugname                                         reaction  pair_count  fatal_count  fatality_rate_pct
  VEDOLIZUMAB                                    Off Label Use        2139           21               0.98
     FENTANYL                                       Drug Abuse        2132         1026              48.12
   INFLIXIMAB                                    Off Label Use        1915           22               1.15
      ELIQUIS                                            Death        1453          721              49.62
     HAEGARDA                            Hereditary Angioedema        1426            0               0.00
   INFLIXIMAB                             Condition Aggravated        1331           10               0.75
   INFLIXIMAB                    Intentional Product Use Issue        1192           12               1.01
     REMICADE                                    Off Label Use        1144            6    

In [8]:
query_path = r"C:\Users\SHRUTI VAISHNAV\OneDrive\Desktop\data science\fda-faers-adverse-event-analysis\SQL"

df_q1.to_csv(os.path.join(query_path, "q1_top_drugs_by_reports.csv"), index=False)
df_q2.to_csv(os.path.join(query_path, "q2_top_drugs_by_fatality.csv"), index=False)
df_q3.to_csv(os.path.join(query_path, "q3_fatality_by_age.csv"), index=False)
df_q4.to_csv(os.path.join(query_path, "q4_top_reactions.csv"), index=False)
df_q5.to_csv(os.path.join(query_path, "q5_drug_reaction_pairs.csv"), index=False)

print("All 5 query results exported to SQL folder:")
print(os.listdir(query_path))

All 5 query results exported to SQL folder:
['faers.db', 'q1_top_drugs_by_reports.csv', 'q2_top_drugs_by_fatality.csv', 'q3_fatality_by_age.csv', 'q4_top_reactions.csv', 'q5_drug_reaction_pairs.csv', 'queries.sql']


In [ ]:
print("SQL ANALYSIS COMPLETE")

print(f"Total records analysed : {len(merged):,}")
print(f"Unique drugs            : {merged['drugname'].nunique():,}")
print(f"Unique reactions        : {merged['reaction'].nunique():,}")
print(f"Fatal cases             : {merged['is_fatal'].sum():,}")
print(f"Overall fatality rate   : {merged['is_fatal'].mean()*100:.2f}%")
print(f"Query results saved to  : {query_path}")

SQL ANALYSIS COMPLETE
Total records analysed : 1,052,002
Unique drugs            : 3,525
Unique reactions        : 10,602
Fatal cases             : 74,839
Overall fatality rate   : 7.11%
Query results saved to  : C:\Users\SHRUTI VAISHNAV\OneDrive\Desktop\data science\fda-faers-adverse-event-analysis\SQL
